In [0]:
%pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.2/925.2 kB 11.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np

# Load the dataset using the fast calamine engine
FILE_PATH = "/Volumes/workspace/default/walmart_data/walmart_supply_chain_500k_clean11.xlsx"

df = pd.read_excel(FILE_PATH, engine="calamine")

print(f"✅ Dataset loaded successfully!")
print(f"   Rows    : {df.shape[0]:,}")
print(f"   Columns : {df.shape[1]}")

✅ Dataset loaded successfully!
   Rows    : 500,000
   Columns : 39


In [0]:

df.head(5)

,data_tier,event_id,event_ts,event_type,lifecycle_stage,scenario_name,channel_path,supplier_name,sku_id,sku_family,brand,product_category,warehouse_id,warehouse_region,retail_store_id,customer_id,customer_region,order_id,order_line_id,shipment_id,rma_id,carrier,qty,unit_cost,sell_price,inventory_status_after,order_status_after,shipment_status_after,finalization_status_after,reverse_status_after,actual_cycle_days,delay_days,exception_code,root_cause_category,revenue_leakage_flag,customer_impact_score,inventory_delta,on_time_flag,anomaly_flag
0,HISTORICAL_SUPPLY_CHAIN,EVT00000001,2025-04-10 04:49:00,RESTOCKED,REVERSE_LOGISTICS,STORE_RETURN_TO_DC,STORE>DC,HP Inc.,HP_LT15,15-inch Laptop,HP,Electronics,DC_WE_CA,West,WM_WE_02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,380.0,549.00,AVAILABLE,NaN,NaN,NaN,RESTOCKED,NaN,NaN,STORE_RETURN,STORE_OVERSTOCK,0,0,10,NaN,1
1,HISTORICAL_SUPPLY_CHAIN,EVT00000002,2023-09-05 00:42:00,STORE_TRANSFER_IN,FORWARD_LOGISTICS,DC_TO_STORE_TRANSFER,DC>STORE,Walmart Private Label,GV_MILK_1GAL,Whole Milk 1 Gallon,Great Value,Grocery,DC_CE_TX,Central,WM_CE_01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12,2.1,3.68,STORE_AVAILABLE,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0,0,12,1.0,0
2,HISTORICAL_SUPPLY_CHAIN,EVT00000003,2018-04-03 14:06:00,CYCLE_COUNT_ADJUSTMENT,INVENTORY_MANAGEMENT,INVENTORY_RECON,DC,Walmart Private Label,GV_MILK_1GAL,Whole Milk 1 Gallon,Great Value,Grocery,DC_WE_CA,West,WM_WE_01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,2.1,3.68,AVAILABLE,NaN,NaN,NaN,NaN,NaN,NaN,CYCLE_COUNT_VAR,INVENTORY_RECON,0,0,9,NaN,1
3,HISTORICAL_SUPPLY_CHAIN,EVT00000004,2018-10-02 07:55:00,SCRAPPED,REVERSE_LOGISTICS,AGED_STOCK_SCRAP,DC,Walmart Private Label,GV_MILK_1GAL,Whole Milk 1 Gallon,Great Value,Grocery,DC_WE_CA,West,WM_WE_03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,181,2.1,3.68,NaN,NaN,NaN,NaN,SCRAPPED,NaN,NaN,AGED_STOCK,OBSOLESCENCE,0,0,-181,NaN,1
4,HISTORICAL_SUPPLY_CHAIN,EVT00000005,2017-10-12 13:44:00,DC_RECEIVED,INVENTORY_MANAGEMENT,SUPPLIER_TO_DC_DELAYED,SUPPLIER>DC,Walmart Private Label,GV_MILK_1GAL,Whole Milk 1 Gallon,Great Value,Grocery,DC_NE_PA,Northeast,WM_NE_03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,202,2.1,3.68,AVAILABLE,NaN,NaN,NaN,NaN,12.0,5.0,SUPPLIER_DELAY,SUPPLIER_CAPACITY,0,0,202,0.0,1


In [0]:
# This tells us two things:
# 1. What every column is called (exact spelling matters when writing code)
# 2. What data type pandas assigned to each column
#
# WHY THIS MATTERS: If a date column shows as 'object' instead of 'datetime',
# or a number shows as 'object', it means pandas misread it.
# We need to fix those before doing any analysis.

print("COLUMN NAMES & DATA TYPES")
print("=" * 45)
for col in df.columns:
    print(f"  {col:<35} {str(df[col].dtype)}")

COLUMN NAMES & DATA TYPES
  data_tier                           object
  event_id                            object
  event_ts                            datetime64[ns]
  event_type                          object
  lifecycle_stage                     object
  scenario_name                       object
  channel_path                        object
  supplier_name                       object
  sku_id                              object
  sku_family                          object
  brand                               object
  product_category                    object
  warehouse_id                        object
  warehouse_region                    object
  retail_store_id                     object
  customer_id                         object
  customer_region                     object
  order_id                            object
  order_line_id                       float64
  shipment_id                         object
  rma_id                              object
  carrier           

In [0]:
# WHY: Missing data is one of the most common issues in real-world datasets.
# But the key mindset: not all nulls are errors.
# Some columns are SUPPOSED to be null for certain event types.
# A junior analyst always asks WHY before deciding what to do with nulls.

null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_summary = pd.DataFrame({
    'missing_count': null_counts,
    'missing_%': null_pct
}).sort_values('missing_%', ascending=False)

# Only show columns that actually have nulls
null_summary = null_summary[null_summary['missing_count'] > 0]

print("MISSING VALUE ANALYSIS")
print("=" * 45)
print(null_summary.to_string())

MISSING VALUE ANALYSIS
                           missing_count  missing_%
reverse_status_after              432224       86.4
rma_id                            405423       81.1
carrier                           368131       73.6
order_status_after                320325       64.1
root_cause_category               312974       62.6
exception_code                    312974       62.6
retail_store_id                   279818       56.0
inventory_status_after            209663       41.9
shipment_id                       180001       36.0
order_line_id                     180000       36.0
order_id                          180000       36.0
customer_region                   180000       36.0
customer_id                       180000       36.0
finalization_status_after         180000       36.0
shipment_status_after             180000       36.0
actual_cycle_days                 158542       31.7
on_time_flag                       68742       13.7
delay_days                         50743 

## Missing Value Analysis — Findings

All null values in this dataset are **structural nulls** — they are null by design,
not because of data quality issues. Each column only applies to specific event types:

- `rma_id`, `reverse_status_after` → only for return events
- `carrier`, `shipment_id` → only for shipment events  
- `exception_code`, `root_cause_category` → only when something went wrong
- `order_id`, `customer_id` → only for customer-facing events

**Decision: We will NOT drop any rows.**  
Instead, we will filter by lifecycle stage or event type when analysing specific areas.

In [0]:
# WHY: For text columns we need to know:
# 1. How many unique values exist?
# 2. What are the most common ones?
# 3. Is the data balanced or does one value dominate?
#
# This shapes how we analyse later — if one category dominates,
# overall averages will be biased toward it.

cat_cols = [
    'lifecycle_stage',
    'product_category', 
    'warehouse_region',
    'channel_path',
    'event_type',
    'brand',
    'exception_code',
    'root_cause_category'
]

for col in cat_cols:
    print(f"\n{'='*55}")
    print(f"  {col.upper()}  ({df[col].nunique()} unique values)")
    print(f"{'='*55}")
    vc = df[col].value_counts(dropna=True)
    for val, count in vc.head(6).items():
        bar = '█' * int(count / len(df) * 50)
        pct = count / len(df) * 100
        print(f"  {str(val):<38} {count:>7,}  ({pct:.1f}%) {bar}")
    if df[col].nunique() > 6:
        print(f"  ... +{df[col].nunique() - 6} more values")


  LIFECYCLE_STAGE  (7 unique values)
  INVENTORY_MANAGEMENT                   117,076  (23.4%) ███████████
  SHIPMENT                               100,855  (20.2%) ██████████
  FULFILLMENT                             79,343  (15.9%) ███████
  FORWARD_LOGISTICS                       71,176  (14.2%) ███████
  REVERSE_LOGISTICS                       67,776  (13.6%) ██████
  ORDERS                                  45,775  (9.2%) ████
  ... +1 more values

  PRODUCT_CATEGORY  (5 unique values)
  Grocery                                281,937  (56.4%) ████████████████████████████
  Electronics                            113,496  (22.7%) ███████████
  Household Essentials                    55,349  (11.1%) █████
  Apparel                                 32,448  (6.5%) ███
  Health & Wellness                       16,770  (3.4%) █

  WAREHOUSE_REGION  (4 unique values)
  Northeast                              140,808  (28.2%) ██████████████
  Southeast                              133,673  (

In [0]:
# WHY: For numeric columns we look at the range, average, and spread.
# The key thing to spot is the difference between mean and median —
# a big gap tells you there are outliers pulling the average up.

num_cols = [
    'qty', 
    'unit_cost', 
    'sell_price',
    'actual_cycle_days', 
    'delay_days',
    'customer_impact_score', 
    'inventory_delta'
]

print("NUMERIC COLUMNS — SUMMARY STATISTICS")
print("=" * 60)
print(df[num_cols].describe().round(2).to_string())

NUMERIC COLUMNS — SUMMARY STATISTICS
             qty  unit_cost  sell_price  actual_cycle_days  delay_days  customer_impact_score  inventory_delta
count  500000.00  500000.00   500000.00          341458.00   449257.00              500000.00        500000.00
mean       29.91      63.08       93.02               3.39        0.98                  20.32            20.67
std        67.30     122.41      175.71               4.42        2.08                  24.10            70.68
min         0.00       0.30        0.58               0.00        0.00                   0.00          -499.00
25%         1.00       2.10        3.68               0.00        0.00                   0.00             0.00
50%         1.00       2.80        4.12               2.00        0.00                  15.00             0.00
75%        10.00      10.00       18.48               4.00        2.00                  20.00             4.00
max       500.00     380.00      549.00              19.00       14.00     

In [0]:
# WHY: These three columns are the heart of our operational efficiency analysis.
# They directly measure how well the supply chain is performing.
# We need to understand exactly how many events are flagged and how many are not.

flag_cols = ['on_time_flag', 'anomaly_flag', 'revenue_leakage_flag']

print("PERFORMANCE FLAGS — VALUE COUNTS")
print("=" * 50)

for col in flag_cols:
    vc = df[col].value_counts(dropna=False)
    total = len(df)
    print(f"\n  {col.upper()}")
    print(f"  {'-'*40}")
    for val, count in vc.items():
        bar = '█' * int(count / total * 40)
        pct = count / total * 100
        label = str(val)
        print(f"  {label:<10} {count:>7,}  ({pct:.1f}%)  {bar}")

PERFORMANCE FLAGS — VALUE COUNTS

  ON_TIME_FLAG
  ----------------------------------------
  1.0        299,484  (59.9%)  ███████████████████████
  0.0        131,774  (26.4%)  ██████████
  nan         68,742  (13.7%)  █████

  ANOMALY_FLAG
  ----------------------------------------
  0          312,974  (62.6%)  █████████████████████████
  1          187,026  (37.4%)  ██████████████

  REVENUE_LEAKAGE_FLAG
  ----------------------------------------
  0          465,031  (93.0%)  █████████████████████████████████████
  1           34,969  (7.0%)  ██


In [0]:
# WHY: Understanding the time span tells you whether trends are meaningful.
# 10 years of data means we can look at year-over-year performance —
# is the supply chain getting better or worse over time?

print("DATE RANGE ANALYSIS")
print("=" * 40)
print(f"  Earliest : {df['event_ts'].min()}")
print(f"  Latest   : {df['event_ts'].max()}")
print(f"  Span     : {(df['event_ts'].max() - df['event_ts'].min()).days:,} days")

print(f"\n  EVENTS PER YEAR")
print(f"  {'-'*30}")
yearly = df['event_ts'].dt.year.value_counts().sort_index()
for year, count in yearly.items():
    bar = '█' * int(count / yearly.max() * 30)
    print(f"  {year}  {count:>6,}  {bar}")

DATE RANGE ANALYSIS
  Earliest : 2016-01-01 01:36:00
  Latest   : 2026-01-13 02:50:00
  Span     : 3,665 days

  EVENTS PER YEAR
  ------------------------------
  2016  18,127  ███
  2017  18,057  ███
  2018  17,732  ██
  2019  17,853  ███
  2020  18,249  ███
  2021  17,880  ███
  2022  18,066  ███
  2023  17,988  ███
  2024  177,223  █████████████████████████████
  2025  178,438  ██████████████████████████████
  2026     387  


## 📊 Data Understanding — Key Findings Summary

| Column | Finding | Implication |
|---|---|---|
| Shape | 500,000 rows × 39 columns | Large dataset, needs efficient filtering |
| event_ts | 2016–2026, 10 year span | Focus on 2024–2025 where data is richest |
| Nulls | All nulls are structural | Do NOT drop rows — filter by event type instead |
| on_time_flag | Only 59.9% on time | Core problem to investigate |
| anomaly_flag | 37.4% flagged | Unusually high — needs root cause analysis |
| revenue_leakage_flag | 7% flagged (34,969 events) | Small % but significant at Walmart's scale |
| product_category | Grocery = 56% | Always segment by category to avoid bias |
| delay_days | Median = 0, Mean = 1, Max = 14 | Right-skewed — small number of severe delays |
| exception_code | BACKORDER top failure mode | Supplier issues are the #1 root cause |
| warehouse_region | Fairly balanced across 4 regions | Safe to compare regions directly |

### Decisions going into Step 2 (Data Cleaning):
- Parse event_ts as datetime ✅ (already correct)
- Fix order_line_id and on_time_flag from float to appropriate types
- Create derived columns: margin, is_delayed, year, month
- Focus analysis on 2024–2025 data for trend work
- Never drop nulls — filter contextually instead